# Deep learning models for arrhythmia detection and diagnosis

## Initialisation

In [65]:
# Imports
import os
from os.path import join as osj
import pandas as pd
import numpy as np
import torch
import wfdb
import pywt
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

# Directory where the dataset is stored
dataset_directory = 'mit-bih-arrhythmia-database-1.0.0/'

# Get an array of all ids
record_ids = pd.read_csv('mit-bih-arrhythmia-database-1.0.0/RECORDS', header=None).to_numpy().reshape(-1)

# Initialise dictionaries to store data
lead1 = {}
lead2 = {}
annotations = {}

# Load ECG signals and annotations
for id in record_ids:
    signals, info = wfdb.io.rdsamp(osj(dataset_directory, str(id)))
    annotation = wfdb.rdann(dataset_directory + str(id), "atr")
    lead1[id] = signals[:, 0]
    lead2[id] = signals[:, 1]
    annotations[id] = annotation


## Data Preprocessing

In [64]:
def swt_denoise(signal):
    """
    Denoise signal using Stationary Wavelet Transform (SWT) with soft thresholding.

    Parameters:
        signal (np.ndarray): The input 1D signal.
        wavelet (str): The wavelet to use.

    Returns:
        np.ndarray: The denoised signal.
    """
    # Get max level
    max_level = pywt.swt_max_level(len(signal))
    
    wavelet = "bior3.1"
    
    # Decompose
    coeffs = pywt.swt(signal, wavelet, level=max_level)
    
    # Estimate noise from first detail coefficients
    detail_coeffs = np.array(coeffs[0][1])
    sigma = np.median(np.abs(detail_coeffs)) / 0.6745
    threshold = sigma * np.sqrt(2 * np.log(len(signal)))
    
    # Apply soft thresholding to detail coefficients
    new_coeffs = []
    for approx, detail in coeffs:
        detail = pywt.threshold(detail, threshold, mode='soft')
        new_coeffs.append((approx, detail))
    
    # Reconstruct
    denoised = pywt.iswt(new_coeffs, wavelet)
    return denoised

def normalise_zscore(data):
    """
    Standardises the input.
    
    Parameters:
        data : np.ndarray
        
    Returns:
        standardised data : np.ndarray
    """
    mean_val = np.mean(data)
    std_val = np.std(data)
    return (data - mean_val) / std_val

def segmentation(signal, annotation, keep_first=False,skew=0.66):  
    """ 
    Segments the data from signal based on the locations in annotation and the skew
    
    Parameters:
        signal : np.ndarray
        annotation : an annotation object extracted from the .atr file
        skew : default = 0.66. a float that represents the percentage of samples taken after the annotation. e.g. 0.4 would mean 40% samples taken after and the rest are before.
    """
    r_peaks = annotation.sample
    r_labels = annotation.symbol
    
    segments = []
    labels = []
    
    # Ensure enough R-peaks to segment
    if len(r_peaks) < 2:
        return [], []
    
    # Compute midpoints between R-peaks
    midpoints = [
        int(r_peaks[i] + skew * (r_peaks[i + 1] - r_peaks[i]))
        for i in range(len(r_peaks) - 1)
    ]
    
    # Handle all beats with a midpoint either side (not first and last)
    for i in range(1, len(midpoints)-1):
        start = midpoints[i]
        end = midpoints[i+1]
        segment = signal[start:end]
        label = r_labels[i]  # Label corresponding to the center R-peak
        segments.append(segment)
        labels.append(label)
    
    # Handle the first beat
    if keep_first: 
        first_segment = signal[:midpoints[0]]
        segments.insert(0, first_segment)
        labels.insert(0, r_labels[0])
    
    # Handle the last beat
    if midpoints[-1] < len(signal):
        last_segment = signal[midpoints[-1]:]
        segments.append(last_segment)
        labels.append(r_labels[-1])
    
    return segments, labels
            
def process_signal(id):
    unfiltered, annotation = lead1[id], annotations[id]
    denoised_data = swt_denoise(unfiltered)
    normalised_data = normalise_zscore(denoised_data)
    beats, labels = segmentation(normalised_data, annotation)
    return beats, labels
     

## Models

### Beat Classifier

In [ ]:
class BeatClassifier(nn.Module):
    def __init__(self, input_size, num_classes):
        super(BeatClassifier, self).__init__()
        
        # Feature extraction layers
        self.feature_extractor = nn.Sequential(
            nn.Conv1d(in_channels=1, out_channels=64, kernel_size=5, stride=1, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2),
            nn.Conv1d(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2)
        )
        
        # Fully connected layers for classification
        self.classifier = nn.Sequential(
            nn.Linear(128 * (input_size // 4), 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )
    
    def forward(self, x):
        x = self.feature_extractor(x)
        x = x.view(x.size(0), -1)  # Flatten
        x = self.classifier(x)
        return x

### Signal Classifier

In [ ]:
class SignalClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(SignalClassifier, self).__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x, _ = self.lstm(x)
        x = x[:, -1, :]  # Take the last time step output
        x = self.dropout(x)
        x = self.fc(x)
        return x
    
class SignalTransformer(nn.Module):
    def __init__(self, input_size=3, d_model=64, num_heads=4, num_layers=2, num_classes=5):
        super(SignalTransformer, self).__init__()
        
        # Positional Encoding
        self.positional_encoding = nn.Parameter(torch.zeros(1, 100, d_model))  # Max sequence length = 100
        
        # Transformer Encoder Layer
        self.encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=num_heads)
        self.transformer = nn.TransformerEncoder(self.encoder_layer, num_layers=num_layers)
        
        # Fully Connected Layer
        self.fc = nn.Linear(d_model, num_classes)
        self.softmax = nn.Softmax(dim=1)
        
    def forward(self, x):
        # x: Batch size x sequence length x input size
        x = x * np.sqrt(64)  # Scaling input (similar to Transformer)
        
        # Add positional encoding (optional)
        x = x + self.positional_encoding[:, :x.size(1), :]
        
        # Transformer expects input shape (seq_len, batch_size, input_size)
        x = x.permute(1, 0, 2)  # Shape: (sequence_length, batch_size, feature_size)
        
        # Pass through Transformer Encoder
        x = self.transformer(x)
        
        # Use the output of the last token
        x = x[-1, :, :]  # Only take the last token's output
        
        # Final classification
        x = self.fc(x)
        x = self.softmax(x)
        return x

### Transformer

## Training

In [ ]:
# Training variables

### Training loops

In [ ]:
def train_beat_classifier(model, criterion, optimizer, record_ids):
    model.train()
    total_loss, correct = 0, 0
    for id in record_ids:
        beats, labels = process_signal(id)
        optimizer.zero_grad()
        outputs = model(beats)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        correct += (predicted == labels).sum().item()
        
        # got this far
        
    return total_loss / len(beats), correct / len(beats)

def train_signal_classifier(model, criterion, optimizer, record_ids):
    pass

## Testing

In [ ]:
def evaluate_model(model, dataloader, device):
    model.eval()
    correct = 0
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            correct += (predicted == labels).sum().item()
    return correct / len(dataloader.dataset)

## Evaluation